Training Qmix model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch import nn, tensor
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from torch.distributions import Categorical
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
from tqdm.notebook import tqdm
import pickle
#import torchsummary
import os
from torch.nn import SmoothL1Loss, MSELoss
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import TensorDataset, Sampler
import copy
from torch.utils.data import TensorDataset, DataLoader
import matplotlib
matplotlib.rcParams['font.family'] = 'serif'
pd.set_option('display.max_seq_items', None)

version = 'ver_Final'
action_type = 'MultiAction'
date = version
train_data = pd.read_csv(f'train.csv')
validation_data = pd.read_csv(f'validation.csv')
test_data = pd.read_csv(f'test.csv')

train_shock_ratio = len(train_data[train_data['Target Shock']>1])/len(train_data)
val_shock_ratio = len(validation_data[validation_data['Target Shock']>1])/len(validation_data)

In [ ]:
scale_columns = ['Heart rate', 'Blood pressure systolic', 'Blood pressure diastolic', 'Blood pressure mean', 'Respiratory rate', 'SpO2', 'Temperature', 'Shock index',
            'Heart rate:Delta', 'Blood pressure systolic:Delta', 'Blood pressure diastolic:Delta', 'Blood pressure mean:Delta', 'Respiratory rate:Delta', 'SpO2:Delta', 'Temperature:Delta', 'Shock index:Delta',
            'Lactate', 'Creatinine', 'Hemoglobin',  'Troponin', 'Lactate Elevation index', 'Creatinine Elevation index', 'Hemoglobin Elevation index', 
            'Propofol', 'Midazolam', 'Fentanyl', 'Ketamine', 'Treated_Diuretic:Binary']

scaler = MinMaxScaler()

train_data_scaled_part = scaler.fit_transform(train_data[scale_columns])
train_data_scaled_part = pd.DataFrame(train_data_scaled_part, columns=scale_columns)

train_data_other_columns = train_data.drop(columns=scale_columns)
train_data_scaled1 = pd.concat([train_data_other_columns, train_data_scaled_part], axis=1)

val_data_scaled_part = scaler.transform(validation_data[scale_columns])
val_data_scaled_part = pd.DataFrame(val_data_scaled_part, columns=scale_columns)

val_data_other_columns = validation_data.drop(columns=scale_columns)
val_data_scaled1 = pd.concat([val_data_other_columns, val_data_scaled_part], axis=1)

train_data_scaled = train_data_scaled1.fillna(-2)
validation_data_scaled = val_data_scaled1.fillna(-2)

train_data_scaled['r:TeamReward'] *= 0.01
validation_data_scaled['r:TeamReward'] *= 0.01

In [ ]:
def make_transition_qmix(data):
    df = data
    s_col = ['0', '1', '2', '3', '4', '5', '6', '7',
       '8', '9', '10', '11', '12', '13', '14', '15', '16',
       '17', '18', '19', '20', '21', '22', '23', '24', '25',
       '26', '27', '28', '29', '30', '31', '32', '33', '34',
       '35', '36', '37', '38', '39', '40', '41', '42', '43',
       '44', '45', '46', '47', '48', '49', '50', '51', '52',
       '53', '54', '55', '56', '57', '58', '59', '60', '61',
       '62', '63', '64', '65', '66', '67', '68', '69', '70',
       '71', '72', '73', '74', '75', '76', '77', '78', '79',
       '80', '81', '82', '83', '84', '85', '86', '87', '88',
       '89', '90', '91', '92', '93', '94', '95', '96', '97',
       '98', '99', '100', '101', '102', '103', '104', '105',
       '106', '107', '108', '109', '110', '111', '112', '113',
       '114', '115', '116', '117', '118', '119', '120', '121',
       '122', '123', '124', '125', '126', '127', '128', '129',
       '130', '131', '132', '133', '134', '135', '136', '137',
       '138', '139', '140', '141', '142', '143', '144', '145',
       '146']
    # Actions
    aT_col = [x for x in df.columns if x == 'a:Transfusion']
    aF_col = [x for x in df.columns if x == 'a:LR_NS_AL']
    aV_col = [x for x in df.columns if x == 'a:Vasopressors']

    # Rewards / Terminals / Labels    
    rT_col = [x for x in df.columns if x == 'r:TeamReward']
    
    t_col = [x for x in df.columns if x[:2] == 't:']
    label_col = [x for x in df.columns if x == 'sh:Shock']
    sampling_col = [x for x in df.columns if x == 'sampling:Shock']
    cont_label_col = [x for x in df.columns if x == 'Cont_label']

    # Prev Actions (treated action columns)
    treated_action_T_col = [x for x in df.columns if x == 'Treated_action:Transfusion']
    treated_action_F_col = [x for x in df.columns if x == 'Treated_action:LR_NS_AL']
    treated_action_V_col = [x for x in df.columns if x == 'Treated_action:Vasopressors_mapped']
    
    global_action_col = [x for x in df.columns if x == 'a:Action']

    dict = {}
    dict['traj'] = {}
    data_len = 0

    s, aT, aF, aV, rT, s2, t, label, sampling, cont_label = [], [], [], [], [], [], [], [], [], []
    treated_action_T, treated_action_F, treated_action_V, global_action = [], [], [], []

    for traj in tqdm(df.stay_id.unique()):
        df_traj = df[df['stay_id'] == traj]
        dict['traj'][traj] = {
            's': df_traj[s_col].values.tolist(),
            'aT': df_traj[aT_col].values.tolist(),
            'aF': df_traj[aF_col].values.tolist(),
            'aV': df_traj[aV_col].values.tolist(),
            'rT': df_traj[rT_col].values.tolist(),
            't': df_traj[t_col].values.tolist(),
            'label': df_traj[label_col].values.tolist(),
            'sampling': df_traj[sampling_col].values.tolist(),
            'cont_label': df_traj[cont_label_col].values.tolist(),
            'treated_action_T': df_traj[treated_action_T_col].values.tolist(),
            'treated_action_F': df_traj[treated_action_F_col].values.tolist(),
            'treated_action_V': df_traj[treated_action_V_col].values.tolist(),
            'global_action': df_traj[global_action_col].values.tolist()
        }

        step_len = len(df_traj) - 1
        for step in range(step_len):
            s.append(dict['traj'][traj]['s'][step])
            aT.append(dict['traj'][traj]['aT'][step])
            aF.append(dict['traj'][traj]['aF'][step])
            aV.append(dict['traj'][traj]['aV'][step])
            rT.append(dict['traj'][traj]['rT'][step+1])
            s2.append(dict['traj'][traj]['s'][step+1])
            t.append(dict['traj'][traj]['t'][step+1])
            label.append(dict['traj'][traj]['label'][step+1])
            sampling.append(dict['traj'][traj]['sampling'][step])
            cont_label.append(dict['traj'][traj]['cont_label'][step])
            treated_action_T.append(dict['traj'][traj]['treated_action_T'][step])
            treated_action_F.append(dict['traj'][traj]['treated_action_F'][step])
            treated_action_V.append(dict['traj'][traj]['treated_action_V'][step])
            global_action.append(dict['traj'][traj]['global_action'][step])
            data_len += 1

    s = torch.FloatTensor(np.float32(s))
    aT = torch.LongTensor(np.int64(aT))
    aF = torch.LongTensor(np.int64(aF))
    aV = torch.LongTensor(np.int64(aV))
    rT = torch.FloatTensor(np.float32(rT))
    s2 = torch.FloatTensor(np.float32(s2))
    t = torch.LongTensor(np.int64(t))
    label = torch.LongTensor(np.int64(label))
    sampling = torch.LongTensor(np.int64(sampling))
    cont_label = torch.LongTensor(np.int64(cont_label))
    treated_action_T = torch.LongTensor(np.int64(treated_action_T))
    treated_action_F = torch.LongTensor(np.int64(treated_action_F))
    treated_action_V = torch.LongTensor(np.int64(treated_action_V))
    global_action = torch.LongTensor(np.int64(global_action))

    actions_list = [aT, aF, aV]
    prev_actions_list = [treated_action_T, treated_action_F, treated_action_V]  

    return s, s2, rT, t, actions_list, label, sampling, cont_label, prev_actions_list, global_action


class CustomSampler(Sampler):
    def __init__(self, data, batch_size, ns):
        self.data = data
        self.batch_size = batch_size
        self.num_samples_1 = ns
        self.num_samples_0 = batch_size - ns
        self.indices = [i for i in range(len(data)) if data[i][6].item() == 0]
        self.indices_neg = [i for i in range(len(data)) if data[i][6].item() == 1]
        self.used_indices_neg = []

    def __iter__(self):
        np.random.shuffle(self.indices)
        np.random.shuffle(self.indices_neg)

        batch = []
        for idx in self.indices:
            batch.append(idx)
            if len(batch) == self.num_samples_0:
                batch.extend(self._sample_indices_neg())
                yield batch
                batch = []

    def _sample_indices_neg(self, remaining=0):
        if remaining:
            num_samples_1 = self.batch_size - remaining
        else:
            num_samples_1 = self.num_samples_1

        if len(self.used_indices_neg) + num_samples_1 > len(self.indices_neg):
            self.used_indices_neg = []
            np.random.shuffle(self.indices_neg)

        indices_neg = self.indices_neg[len(self.used_indices_neg):len(self.used_indices_neg) + num_samples_1]
        self.used_indices_neg.extend(indices_neg)
        return indices_neg

    def __len__(self):
        return (len(self.indices) + len(self.indices_neg)) // self.batch_size

In [ ]:
# train_transition
train_s, train_s2, train_reward, train_t, train_actions_list, train_label, train_sampling, train_cont_label, train_prev_actions_list, train_global_action = make_transition_qmix(train_data_scaled)

train_transition = TensorDataset(
    train_s, train_s2, train_reward, train_t,
    torch.stack(train_actions_list, dim=1),
    train_label, train_sampling, train_cont_label,
    torch.stack(train_prev_actions_list, dim=1),
    train_global_action
)

# val_transition
val_s, val_s2, val_reward, val_t, val_actions_list, val_label, val_sampling, val_cont_label, val_prev_actions_list, val_global_action = make_transition_qmix(validation_data_scaled)

val_transition = TensorDataset(
    val_s, val_s2, val_reward, val_t,
    torch.stack(val_actions_list, dim=1),
    val_label, val_sampling, val_cont_label,
    torch.stack(val_prev_actions_list, dim=1),
    val_global_action
)

# val_return_transition
val_return_transition = TensorDataset(
    val_s, val_s2, val_reward, val_t,
    torch.stack(val_actions_list, dim=1),
    val_label, val_sampling, val_cont_label,
    torch.stack(val_prev_actions_list, dim=1),
    val_global_action
)


Qmix

In [ ]:
# QMix Network
# ==============================================
def init_weights(m):
    if type(m) in [nn.Linear, nn.Conv2d]:
        torch.nn.init.kaiming_uniform_(m.weight)
        m.bias.data.fill_(0.)

# ==============================================

class QMixNetwork(nn.Module):
    def __init__(self, num_agents, input_dim, embed_dim=128, num_quantiles=4):
        super(QMixNetwork, self).__init__()
        self.num_agents = num_agents
        self.input_dim = input_dim
        self.embed_dim = embed_dim
        self.num_quantiles = num_quantiles

        # Hypernetworks
        self.hyper_w_1 = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ELU(),
            nn.Linear(embed_dim, num_agents * embed_dim)
        )
        self.hyper_w_final = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ELU(),
            nn.Linear(embed_dim, embed_dim)
        )

        self.hyper_b_1 = nn.Linear(input_dim, embed_dim)
        
        self.hyper_b_final = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ELU(),
            nn.Linear(embed_dim, 1) 
        )
                    
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.hyper_w_final.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=0.01)
                nn.init.zeros_(m.bias)
        for m in self.hyper_b_final.modules():
            if isinstance(m, nn.Linear):
                nn.init.zeros_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, agent_qs, states):
        """
        agent_qs: (bs, n_agents, nq)
        states:   (bs, state_dim)
        """
        bs = agent_qs.size(0)
        nq = agent_qs.size(2)
        n_agents = self.num_agents

        w1 = torch.abs(self.hyper_w_1(states)).view(bs, n_agents, self.embed_dim)
        b1 = self.hyper_b_1(states).view(bs, 1, self.embed_dim) # (bs, 1, embed)

        agent_qs_transposed = agent_qs.transpose(1, 2) # (bs, nq, n_agents)
        hidden = torch.bmm(agent_qs_transposed, w1) + b1 # (bs, nq, embed)
        hidden = F.elu(hidden)

        w_final = torch.abs(self.hyper_w_final(states)).view(bs, self.embed_dim, 1)
        b_final = self.hyper_b_final(states).view(bs, 1, 1) # (bs, 1, 1)

        q_total = torch.bmm(hidden, w_final) + b_final
        
        return q_total.squeeze(2) # (bs, nq)

class CriticHead(nn.Module):
    def __init__(self, state_dim, action_dim, num_quantiles):
        super().__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.num_quantiles = num_quantiles

        if action_dim == 2:
            head_in = state_dim + action_dim
            # feature extractor inside head
            self.shared = nn.Sequential(
                nn.Linear(head_in, head_in),
                nn.LayerNorm(head_in, eps=1e-6, elementwise_affine=True),
                nn.ELU(),
                nn.Linear(head_in, 128),
                nn.LayerNorm(128, eps=1e-6, elementwise_affine=True),
                nn.ELU(),
            )

            # advantage stream -> action_dim * num_quantiles
            self.advantage = nn.Sequential(
                nn.Linear(128, 64),
                nn.ELU(),
                nn.Linear(64, action_dim * num_quantiles)
            )

            # state value (scalar)
            self.state_value = nn.Sequential(
                nn.Linear(128, 32),
                nn.ELU(),
                nn.Linear(32, 1)
            )

            self.apply(init_weights)
            
        elif action_dim == 4:
            head_in = state_dim + action_dim
            self.shared = nn.Sequential(
                nn.Linear(head_in, head_in),
                nn.LayerNorm(head_in, eps=1e-6, elementwise_affine=True),
                nn.ELU(),
                nn.Linear(head_in, 256),
                nn.LayerNorm(256, eps=1e-6, elementwise_affine=True),
                nn.ELU(),
            )

            # advantage stream -> action_dim * num_quantiles
            self.advantage = nn.Sequential(
                nn.Linear(256, 128),
                nn.ELU(),
                nn.Linear(128, action_dim * num_quantiles)
            )

            # state value (scalar)
            self.state_value = nn.Sequential(
                nn.Linear(256, 64),
                nn.ELU(),
                nn.Linear(64, 1)
            )

            self.apply(init_weights)

    def forward(self, shared_h, prev_action):
        # shared_h: [batch, state_dim]
        # prev_action: [batch, 1] (long)
        prev_onehot = F.one_hot(prev_action.squeeze(-1), num_classes=self.action_dim).float()
        x = torch.cat([shared_h, prev_onehot], dim=-1)  # [batch, state_dim+action_dim]
        x = self.shared(x)  # [batch, 128]
        adv = self.advantage(x)  # [batch, action_dim * num_quantiles]
        sv = self.state_value(x)  # [batch, 1]
        # normalize adv
        adv_mean = adv.view(adv.size(0), -1).mean(dim=1, keepdim=True)  # [batch,1]
        adv = adv - adv_mean  # broadcasting
        state_action = sv + adv  # [batch, action_dim * num_quantiles]
        out = state_action.view(-1, self.action_dim, self.num_quantiles)  # [batch, action_dim, num_quantiles]
        return out
    
    def save(self, path):
        torch.save(self.state_dict(), path)

    def load(self, path, device):
        self.load_state_dict(torch.load(path, map_location=device))
        self.to(device)

# ==============================================
# QMIX Agent
# ==============================================
class QMIXAgent:
    def __init__(self, state_dim, action_dim_list, device,
                 num_quantiles=4, learning_rate=0.0001, gamma=0.99, embed_dim=32, cql_rate=1.0):
        
        self.device = device
        self.num_agents = len(action_dim_list)
        self.action_dims = action_dim_list
        self.num_quantiles = num_quantiles
        self.gamma = gamma
        self.cql_rate = cql_rate
        self.tau = ((torch.arange(num_quantiles) + 0.5) / float(num_quantiles)).to(device)
        
        self.critics_main = nn.ModuleList([CriticHead(state_dim, a, num_quantiles).to(device) for a in action_dim_list])
        self.critics_target = nn.ModuleList([copy.deepcopy(c).to(device) for c in self.critics_main])
        for t in self.critics_target:
            t.eval()
        
        self.qmix_net = QMixNetwork(num_agents=self.num_agents, input_dim=state_dim, embed_dim=embed_dim, num_quantiles=num_quantiles).to(device)
        self.qmix_target_net = QMixNetwork(num_agents=self.num_agents, input_dim=state_dim, embed_dim=embed_dim, num_quantiles=num_quantiles).to(device)
        self.qmix_target_net.load_state_dict(self.qmix_net.state_dict())
        self.qmix_target_net.eval()
        
        # optimizers
        transfusion_lr = learning_rate * 0.1
        self.critic_main_optimizers = [
            torch.optim.Adam(
                self.critics_main[i].parameters(),
                lr=(transfusion_lr if i == 0 else learning_rate)
            )
            for i in range(self.num_agents)
        ]
        self.qmix_optimizer = torch.optim.Adam(self.qmix_net.parameters(), lr=learning_rate)

        # schedulers
        self.schedulers_critic = [
            torch.optim.lr_scheduler.CosineAnnealingLR(self.critic_main_optimizers[i], T_max=100, eta_min=1e-6)
            for i in range(self.num_agents)
        ]
        self.qmix_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.qmix_optimizer, T_max=100, eta_min=1e-6)

        self.loss_fn = nn.SmoothL1Loss(reduction='none')

    def save_model(self, path_prefix, version):
        for i in range(self.num_agents):
            torch.save(self.critics_main[i].state_dict(), f"{path_prefix}_critic_{i}_v{version}.pt")
        torch.save(self.qmix_net.state_dict(), f"{path_prefix}_qmix_v{version}.pt")
        print(f"✅ QMIX 모델 저장 완료: {path_prefix}")

    def load_model(self, path_prefix, version):
        for i in range(self.num_agents):
            self.critics_main[i].load_state_dict(torch.load(f"{path_prefix}_critic_{i}_v{version}.pt"))
        self.qmix_net.load_state_dict(torch.load(f"{path_prefix}_qmix_v{version}.pt"))
        print(f"✅ QMIX 모델 로드 완료: {path_prefix}")

    def get_q_values(self, state, prev_actions):
        q_values = []
        for i in range(self.num_agents):
            q_val = self.critics_main[i](state, prev_actions[i])  # (batch, action_dim, num_quantiles)
            q_values.append(q_val)
        return q_values

    def get_total_q(self, state, prev_actions):
        q_values = self.get_q_values(state, prev_actions)

        agent_qs = []
        for i in range(self.num_agents):
            q_mean = q_values[i].mean(dim=2)  # (batch, action_dim)
            max_q, _ = q_mean.max(dim=1, keepdim=True)
            agent_qs.append(max_q)

        agent_qs = torch.cat(agent_qs, dim=1)  # (batch, num_agents)
        q_total = self.qmix_net(agent_qs, state)
        return q_total
    
    def get_state_representation(self, critic, state, prev_action, num_actions):
        prev_action_onehot = F.one_hot(prev_action.squeeze(-1), num_classes=num_actions).float()
        combined_state = torch.cat([state, prev_action_onehot], dim=-1)
        state_representation = critic.shared(combined_state)
        expected_state_value = critic.state_value(state_representation)
        return expected_state_value

    def train_critic_qmix(
            self,
            state,
            actions_list,
            reward,
            next_state,
            terminal,
            prev_actions_list,
            total_env_steps,
            max_grad_norm_normal: float = 1.0,
            max_grad_norm_slow: float = 1.0,
            soft_update_rate: float = 0.001,
            cql_temp: float = 5.0, 
            entropy_coeff: float = 0.1
        ):
        for i in range(self.num_agents):
            self.critics_main[i].train()
            self.critics_target[i].eval()
        self.qmix_net.train()
        self.qmix_target_net.eval()
        
        state = state.to(self.device)
        next_state = next_state.to(self.device)
        reward = reward.to(self.device).view(-1, 1)
        done = terminal.to(self.device).view(-1, 1)
        
        critic_losses, td_losses, cql_losses = [], [], []

        agent_qs = []
        agent_cql_losses = [] 
        
        steps_per_epoch = 15000
        warmup_start = steps_per_epoch * 1 
        warmup_end = steps_per_epoch * 2 
        
        if total_env_steps < warmup_start:
            current_cql_rate = 0.0
        elif total_env_steps < warmup_end:
            scale = (total_env_steps - warmup_start) / (warmup_end - warmup_start)
            current_cql_rate = self.cql_rate * scale
        else:
            current_cql_rate = self.cql_rate
        
        for i in range(self.num_agents):
            prev_action = prev_actions_list[:, i].to(self.device)
            all_q_dist = self.critics_main[i](state, prev_action)
            mean_all_q = all_q_dist.mean(dim=2) 
            
            probs = F.softmax(mean_all_q / cql_temp, dim=1)
            log_probs = F.log_softmax(mean_all_q / cql_temp, dim=1)
            entropy = -(probs * log_probs).sum(dim=1).mean()
            
            logsumexp_q = cql_temp * torch.logsumexp(mean_all_q / cql_temp, dim=1)
            
            action_idx = actions_list[:, i].to(self.device).view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
            q_dist = all_q_dist.gather(1, action_idx).squeeze(1)
            dataset_q = q_dist.mean(dim=1)
            
            raw_cql_loss = logsumexp_q - dataset_q
            
            agent_cql_losses.append(raw_cql_loss.mean() - entropy_coeff * entropy)
            agent_qs.append(q_dist)

        total_cql_loss = torch.stack(agent_cql_losses).sum()

        agent_qs_tensor = torch.stack(agent_qs, dim=1) # (batch, num_agents, nq)
        q_total_main = self.qmix_net(agent_qs_tensor, state)

        with torch.no_grad():
            next_agent_qs = []

            for i in range(self.num_agents):
                curr_action_as_prev = actions_list[:, i].to(self.device)
                next_q_main_dist = self.critics_main[i](next_state, curr_action_as_prev)
                mean_q = next_q_main_dist.mean(dim=2)
                
                if i == 2:
                    max_action_index = torch.zeros(curr_action_as_prev.size(0), dtype=torch.long, device=self.device)
                    restricted_actions = torch.tensor([0, 3], device=self.device)
                    restricted_mask = (curr_action_as_prev.squeeze(-1) == 0)
                    
                    if restricted_mask.any():
                        res_q = mean_q[restricted_mask][:, restricted_actions]
                        res_max_idx = res_q.max(dim=1)[1]
                        max_action_index[restricted_mask] = restricted_actions[res_max_idx]
                    if (~restricted_mask).any():
                        max_action_index[~restricted_mask] = mean_q[~restricted_mask].max(dim=1)[1]
                else:
                    max_action_index = mean_q.max(dim=1)[1]

                next_q_target_dist = self.critics_target[i](next_state, curr_action_as_prev)
                next_act_exp = max_action_index.view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                next_q_i = next_q_target_dist.gather(1, next_act_exp).squeeze(1)
                next_agent_qs.append(next_q_i)
            
            next_agent_qs_tensor = torch.stack(next_agent_qs, dim=1)
            q_total_next = self.qmix_target_net(next_agent_qs_tensor, next_state)
            
            q_target_total = (reward) + (1 - done) * self.gamma * q_total_next
            q_target_total = torch.clamp(q_target_total, min=-100.0, max=100.0)

        # ---------------------------------------------------------
        # 4. 통합 Loss 계산
        # ---------------------------------------------------------
        diff = q_target_total.unsqueeze(2) - q_total_main.unsqueeze(1)
        abs_diff = diff.abs()
        huber_l = torch.where(abs_diff < 1.0, 0.5 * diff ** 2, abs_diff - 0.5)
        
        tau = self.tau.view(1, 1, -1)
        quantile_weight = torch.abs(tau - (diff < 0).float())
        td_loss = (quantile_weight * huber_l).mean(dim=1).sum(dim=1).mean()

        total_loss = td_loss + current_cql_rate * total_cql_loss
        
        td_losses.append(td_loss.item())
        cql_losses.append(total_cql_loss.item())
        critic_losses.append(total_loss.item())

        for opt in self.critic_main_optimizers:
            opt.zero_grad()
        self.qmix_optimizer.zero_grad()

        total_loss.backward()

        for i in range(self.num_agents):
            torch.nn.utils.clip_grad_norm_(self.critics_main[i].parameters(), max_grad_norm_normal)
        torch.nn.utils.clip_grad_norm_(self.qmix_net.parameters(), max_grad_norm_normal)

        for i in range(self.num_agents):
            self.critic_main_optimizers[i].step()
            self.schedulers_critic[i].step()
        self.qmix_optimizer.step()
        self.qmix_scheduler.step()

        with torch.no_grad():
            for i in range(self.num_agents):
                for t_p, m_p in zip(self.critics_target[i].parameters(), self.critics_main[i].parameters()):
                    t_p.data.copy_(soft_update_rate * m_p.data + (1.0 - soft_update_rate) * t_p.data)
            for t_p, m_p in zip(self.qmix_target_net.parameters(), self.qmix_net.parameters()):
                t_p.data.copy_(soft_update_rate * m_p.data + (1.0 - soft_update_rate) * t_p.data)

        return critic_losses, td_losses, cql_losses

    def validate_batch_loss_qmix(self, state, actions_list, reward, next_state, terminal, prev_actions_list):
        with torch.no_grad():
            for i in range(self.num_agents):
                self.critics_main[i].eval()
                self.critics_target[i].eval()
            self.qmix_net.eval()
            self.qmix_target_net.eval()
            
            state = state.to(self.device)
            next_state = next_state.to(self.device)
            reward = reward.to(self.device).view(-1, 1)
            done = terminal.to(self.device).view(-1, 1)
            critic_losses = []

            agent_qs = []
            for i in range(self.num_agents):
                action = actions_list[:, i].to(self.device).view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                prev_action = prev_actions_list[:, i].to(self.device)
                
                q_dist = self.critics_main[i](state, prev_action)
                q_i = q_dist.gather(1, action).squeeze(1)
                agent_qs.append(q_i)

            agent_qs_tensor = torch.stack(agent_qs, dim=1)
            q_total_main = self.qmix_net(agent_qs_tensor, state)

            next_agent_qs = []
            for i in range(self.num_agents):
                curr_action_as_prev = actions_list[:, i].to(self.device)

                next_q_main_dist = self.critics_main[i](next_state, curr_action_as_prev)
                mean_q = next_q_main_dist.mean(dim=2)

                if i == 2:
                    max_action_index = torch.zeros(curr_action_as_prev.size(0), dtype=torch.long, device=self.device)
                    restricted_actions = torch.tensor([0, 3], device=self.device)
                    restricted_mask = (curr_action_as_prev.squeeze(-1) == 0)
                    
                    if restricted_mask.any():
                        res_q = mean_q[restricted_mask][:, restricted_actions]
                        res_max_idx = res_q.max(dim=1)[1]
                        max_action_index[restricted_mask] = restricted_actions[res_max_idx]
                    
                    if (~restricted_mask).any():
                        max_action_index[~restricted_mask] = mean_q[~restricted_mask].max(dim=1)[1]
                else:
                    max_action_index = mean_q.max(dim=1)[1]

                next_q_target_dist = self.critics_target[i](next_state, curr_action_as_prev)
                next_act_exp = max_action_index.view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                next_q_i = next_q_target_dist.gather(1, next_act_exp).squeeze(1)
                next_agent_qs.append(next_q_i)
            
            next_agent_qs_tensor = torch.stack(next_agent_qs, dim=1)
            q_total_next = self.qmix_target_net(next_agent_qs_tensor, next_state)
            
            q_target_total = (reward) + (1 - done) * self.gamma * q_total_next
            q_target_total = torch.clamp(q_target_total, min=-100.0, max=100.0)
            
            diff = q_target_total.unsqueeze(2) - q_total_main.unsqueeze(1)
            abs_diff = diff.abs()
            huber_l = torch.where(abs_diff < 1.0, 0.5 * diff ** 2, abs_diff - 0.5)
            
            tau = self.tau.view(1, 1, -1)
            quantile_weight = torch.abs(tau - (diff < 0).float())
            quantile_huber_loss = (quantile_weight * huber_l).mean(dim=1).sum(dim=1).mean()
            critic_losses.append(quantile_huber_loss.item())
            
            return critic_losses

    def validate_batch_return_qmix(self, state, prev_actions_list):
        with torch.no_grad():
            for i in range(self.num_agents):
                self.critics_main[i].eval()
            self.qmix_net.eval()
            
            state = state.to(self.device)
            batch_size = state.size(0)
            agent_qs = [] 

            for i in range(self.num_agents):
                q_values = self.critics_main[i](state, prev_actions_list[:, i])
                prev_action = prev_actions_list[:, i]

                if i == 2:
                    restricted_actions = torch.tensor([0, 3], device=self.device)
                    restricted_prev_actions = torch.tensor([0], device=self.device)
                    restricted_mask = torch.isin(prev_action, restricted_prev_actions).squeeze()

                    mean_q = q_values.mean(dim=2) # [B, action_dim]
                    _, max_action_index = torch.max(mean_q, dim=1) # [B]

                    if restricted_mask.any():
                        restricted_q_mean = q_values[restricted_mask][:, restricted_actions, :].mean(dim=2)
                        _, restricted_max_idx = torch.max(restricted_q_mean, dim=1)
                        max_action_index[restricted_mask] = restricted_actions[restricted_max_idx]

                else:
                    mean_q = q_values.mean(dim=2)
                    _, max_action_index = torch.max(mean_q, dim=1)

                max_action_index_expanded = max_action_index.view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                agent_q = q_values.gather(1, max_action_index_expanded).squeeze(1)
                agent_qs.append(agent_q)

            agent_qs_tensor = torch.stack(agent_qs, dim=1) 
            joint_q_dist = self.qmix_net(agent_qs_tensor, state)
            joint_q = joint_q_dist.mean(dim=1)
            average_return = joint_q.mean().item()
            return average_return

    def print_policy_restricted_qmix(self, state, actions_list, prev_actions_list):
        with torch.no_grad():
            for i in range(self.num_agents):
                self.critics_main[i].eval()
            self.qmix_net.eval()

            state = state.to(self.device)
            
            clinician_action_indices = []
            clinician_state_action_values_list = [] 
            
            max_action_indices = []
            rl_max_state_action_values_list = []
            
            median_action_indices = []
            rl_median_state_action_values_list = [] 
            
            system_action_indices = []
            state_values = []
            
            for i in range(self.num_agents):
                cl_action = actions_list[:, i].to(self.device)
                prev_action = prev_actions_list[:, i].to(self.device)

                q_values = self.critics_main[i](state, prev_action)
                mean_q = q_values.mean(dim=2)

                # [Clinician]
                clinician_action_indices.append(cl_action)
                cl_idx_exp = cl_action.view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                clinician_state_action_values_list.append(q_values.gather(1, cl_idx_exp).squeeze(1))

                # [RL Max]
                max_idx = mean_q.argmax(dim=1)
                
                # [Intervention threshold]
                sorted_idx = mean_q.argsort(dim=1)
                if i == 0:
                    median_idx = max_idx.clone()
                elif i == 1:
                    median_idx = sorted_idx[:, 2]
                elif i == 2:
                    median_idx = sorted_idx[:, 2]

                if i == 2:
                    restricted_mask = (prev_action.flatten() == 0)
                    if restricted_mask.any():
                        res_actions = torch.tensor([0, 3], device=self.device)
                        res_q_subset = mean_q[restricted_mask]
                        
                        if res_q_subset.dim() == 1:
                            res_q_subset = res_q_subset.unsqueeze(0)
                        
                        res_q_only = torch.index_select(res_q_subset, dim=1, index=res_actions)
                        res_q_avg = res_q_only.mean(dim=1, keepdim=True)
                        target_q_tensor = res_q_subset

                        if target_q_tensor.dim() == 1:
                            target_q_tensor = target_q_tensor.unsqueeze(0)

                        cl_actions_for_mask = clinician_action_indices[i][restricted_mask].long()

                        if cl_actions_for_mask.dim() == 1:
                            cl_actions_for_mask = cl_actions_for_mask.unsqueeze(-1)

                        cl_q_values = torch.gather(target_q_tensor, dim=1, index=cl_actions_for_mask)
                        _, res_max_idx_raw = torch.max(res_q_only, dim=1)
                        rl_best_action = res_actions[res_max_idx_raw]
                        should_switch = (cl_q_values <= res_q_avg).flatten()
                        final_idx = cl_actions_for_mask.flatten()
                        final_idx[should_switch] = rl_best_action[should_switch]
                        median_idx[restricted_mask] = final_idx
                        max_idx[restricted_mask] = rl_best_action

                max_action_indices.append(max_idx)
                median_action_indices.append(median_idx)
                
                med_idx_exp = median_idx.view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                rl_median_state_action_values_list.append(q_values.gather(1, med_idx_exp).squeeze(1))

                max_idx_exp = max_idx.view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                rl_max_state_action_values_list.append(q_values.gather(1, max_idx_exp).squeeze(1))

                state_values.append(self.get_state_representation(self.critics_main[i], state, prev_action, self.action_dims[i]))

            for i in range(self.num_agents):
                cl_m = clinician_state_action_values_list[i].mean(dim=1)
                rl_max_m = rl_max_state_action_values_list[i].mean(dim=1)
                rl_med_m = rl_median_state_action_values_list[i].mean(dim=1)
                
                if i == 0:
                    threshold = (cl_m + rl_max_m) * 0.25
                    sys_idx = torch.where(cl_m <= threshold, max_action_indices[i], clinician_action_indices[i])
                else: 
                    sys_idx = torch.where(cl_m < rl_med_m, max_action_indices[i], clinician_action_indices[i])
                
                system_action_indices.append(sys_idx)

            cl_agent_qs = torch.stack(clinician_state_action_values_list, dim=1)
            clinician_qmix_dist = self.qmix_net(cl_agent_qs, state)
            clinician_qmix_mean = clinician_qmix_dist.mean(dim=1)

            rl_agent_qs = torch.stack(rl_max_state_action_values_list, dim=1)
            rl_max_qmix_dist = self.qmix_net(rl_agent_qs, state)
            rl_max_qmix_mean = rl_max_qmix_dist.mean(dim=1)

            sys_agent_qs_list = []
            for i in range(self.num_agents):
                sys_idx_exp = system_action_indices[i].view(-1, 1, 1).expand(-1, 1, self.num_quantiles)
                q_values_all = self.critics_main[i](state, prev_actions_list[:, i].to(self.device))
                sys_agent_qs_list.append(q_values_all.gather(1, sys_idx_exp).squeeze(1))
            
            system_agent_qs = torch.stack(sys_agent_qs_list, dim=1)
            system_qmix_dist = self.qmix_net(system_agent_qs, state)
            system_qmix_mean = system_qmix_dist.mean(dim=1)

            return (
                clinician_action_indices, clinician_state_action_values_list, [clinician_qmix_dist], [clinician_qmix_mean],
                max_action_indices, rl_max_state_action_values_list, [rl_max_qmix_dist], [rl_max_qmix_mean],
                system_action_indices, [system_qmix_dist], [system_qmix_mean],
                median_action_indices, rl_median_state_action_values_list, state_values
            )
        
def train_and_evaluate_model_qmix(networks: "QMIXAgent", train_transition, val_transition, val_return_transition,
                                  val_return_transition_len, batch_size, epoch_num, cql_rate):
    torch.manual_seed(0)
    np.random.seed(0)

    # ===== 모델 구조 출력 ===== #
    print("\n===== QMIXAgent Model Structure =====")
    for agent_idx, critic in enumerate(networks.critics_main):
        print(f"\n--- Agent {agent_idx} Critic Network ---")
        print(critic)
    print("=====================================\n")
    
    print(f"\n--- Qmix Network ---")
    print(networks.qmix_net)
    print("=====================================\n")

    sampler = CustomSampler(data=train_transition, batch_size=batch_size, ns=int(batch_size * train_shock_ratio))
    train_data_loader = DataLoader(train_transition, batch_sampler=sampler, num_workers=4, pin_memory=True)
    val_data_loader = DataLoader(dataset=val_transition, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    val_return_data_loader = DataLoader(val_return_transition, batch_size=val_return_transition_len, shuffle=False, num_workers=4, pin_memory=True)

    epoch_critic_loss = []
    epoch_td_loss = []
    epoch_cql_loss = []
    epoch_val_critic_loss = []
    epoch_val_return = []

    best_return = -float('inf')
    total_env_steps = 0
    patience_value = 0
    patience_max = 200

    for epoch in tqdm(range(1, epoch_num+1)):
        batch_critic_loss = []
        batch_td_loss = []
        batch_cql_loss = []

        # ============ Train ============ #
        for i_batch, batch_data in enumerate(train_data_loader):
            batch_data = [x.to(networks.device) for x in batch_data]
            train_states, train_next_states, train_rewards, train_terminal, train_actions, train_label, train_sampling, train_cont_label, train_prev_actions, train_global_action = batch_data

            total_env_steps += len(train_rewards)

            state = train_states.to(networks.device)
            next_state = train_next_states.to(networks.device)

            critic_losses, td_losses, cql_losses = networks.train_critic_qmix(
                state, train_actions, train_rewards, next_state, train_terminal, train_prev_actions, total_env_steps
            )

            batch_critic_loss.append(np.sum(critic_losses))
            batch_td_loss.append(np.sum(td_losses))
            batch_cql_loss.append(np.sum(cql_losses))

        # ============ Validation ============ #
        val_batch_critic_loss = []

        for val_i_batch, val_batch_data in enumerate(val_data_loader):
            val_batch_data = [x.to(networks.device) for x in val_batch_data]
            val_states, val_next_states, val_rewards, val_terminal, val_actions, val_label, val_sampling, val_cont_label, val_prev_actions, val_global_action = val_batch_data

            state = val_states.to(networks.device)
            next_state = val_next_states.to(networks.device)

            critic_losses = networks.validate_batch_loss_qmix(
                state, val_actions, val_rewards, next_state, val_terminal, val_prev_actions
            )

            val_batch_critic_loss.append(np.sum(critic_losses))

        val_batch_return = []
        for val_i_batch_return, val_batch_data_return in enumerate(val_return_data_loader):
            val_batch_data_return = [x.to(networks.device) for x in val_batch_data_return]
            val_return_states, val_return_next_states, val_return_rewards, val_return_terminal, val_return_actions, val_return_label, val_return_sampling, val_return_cont_label, val_return_prev_actions, val_return_global_action = val_batch_data_return

            state = val_return_states.to(networks.device)
            total_val_return = networks.validate_batch_return_qmix(state, val_return_prev_actions)
            val_batch_return.append(total_val_return)

        # ============ Logging ============ #
        batch_critic_loss_mean = np.mean(batch_critic_loss)
        batch_td_loss_mean = np.mean(batch_td_loss)
        batch_cql_loss_mean = np.mean(batch_cql_loss)
        batch_val_critic_loss_mean = np.mean(val_batch_critic_loss)
        batch_val_return_mean = np.mean(val_batch_return)

        print(f'Epoch {epoch} critic_loss : {batch_critic_loss_mean}')
        print(f'Epoch {epoch} td_loss : {batch_td_loss_mean}')
        print(f'Epoch {epoch} cql_loss : {batch_cql_loss_mean}')
        print(f'Epoch {epoch} val_critic_loss : {batch_val_critic_loss_mean}')
        print(f'Epoch {epoch} val_return : {batch_val_return_mean}')
        print('------------------------------------------------------------')

        epoch_critic_loss.append(batch_critic_loss_mean)
        epoch_td_loss.append(batch_td_loss_mean)
        epoch_cql_loss.append(batch_cql_loss_mean)
        epoch_val_critic_loss.append(batch_val_critic_loss_mean)
        epoch_val_return.append(batch_val_return_mean)

        critic_loss_value = float(f"{epoch_val_critic_loss[-1]:.4f}")
        return_value = float(f"{epoch_val_return[-1]:.4f}")
        path_prefix = f'RL_model'
        networks.save_model(path_prefix, version)

        if epoch_val_return[-1] > best_return:
            best_return = epoch_val_return[-1]
            patience_value = 0
            
        else:
            patience_value += 1
            if patience_value >= patience_max:
                print(f"Early stopping triggered at epoch {epoch}")
                break
        if (epoch % 50) == 0:
            # ============ Plotting ============ #
            x = list(range(1, len(epoch_critic_loss) + 1))
            plt.figure(figsize=(15, 6))

            plt.subplot(1, 2, 1)
            plt.plot(x, epoch_val_critic_loss, marker='o', label='Epoch Val Critic Loss', color='blue')
            plt.title("Epoch Val Critic Loss")
            plt.xlabel("epoch"); plt.ylabel("Val Critic Loss"); plt.legend()

            plt.subplot(1, 2, 2)
            plt.plot(x, epoch_val_return, marker='o', label='Epoch Val Return', color='blue')
            plt.title("Epoch Val Return")
            plt.xlabel("epoch"); plt.ylabel("Val Return"); plt.legend()

            plt.show()
            
    # ============ Plotting ============ #
    x = list(range(1, len(epoch_critic_loss) + 1))
    plt.figure(figsize=(15, 6))

    plt.subplot(1, 2, 1)
    plt.plot(x, epoch_val_critic_loss, marker='o', label='Epoch Val Critic Loss', color='blue')
    plt.title("Epoch Val Critic Loss")
    plt.xlabel("epoch"); plt.ylabel("Val Critic Loss"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(x, epoch_val_return, marker='o', label='Epoch Val Return', color='blue')
    plt.title("Epoch Val Return")
    plt.xlabel("epoch"); plt.ylabel("Val Return"); plt.legend()

    plt.show()

    epoch_val_return_mean = np.mean(epoch_val_return)
    epoch_val_critic_loss_mean = np.mean(epoch_val_critic_loss)
    print('Total val_return_mean: ', epoch_val_return_mean)

    return epoch_val_critic_loss_mean


In [ ]:
# ✅ 기존 코드
USE_GPU = torch.cuda.is_available()
device = torch.device('cuda:0' if USE_GPU else 'cpu')
print(device)

batch_size = 200
learning_rate = 0.00001
epoch_num = 40

state_dim = 147
action_dim_list = [2, 4, 4]  # Transfusion, Fluids, Vasopressors
cql_value = 0.0001

network_critic = QMIXAgent(
    state_dim=state_dim,
    action_dim_list=action_dim_list,
    device=device,
    learning_rate=learning_rate,
    num_quantiles=5,
    gamma=0.99,
    cql_rate = cql_value
)

result_critic = train_and_evaluate_model_qmix(
    network_critic,
    train_transition,
    val_transition,
    val_return_transition,
    len(validation_data_scaled),
    batch_size,
    epoch_num,
    cql_rate = cql_value
)